# MCP Explorer — Transports & Primitives
### Interactive, cell-by-cell demos | Teal Trust Workshop

Last week we covered the **theory** of MCP as tooling for agentic AI. This notebook is the
**hands-on companion** to `app.py` — the same eight demos, but as runnable cells you step through
live in class instead of a web UI.

**The big idea in one line:** MCP uses **one** standard wire format — **JSON-RPC 2.0** — carried
over different **transports** (how a message travels) to deliver five **primitives** (what a
message carries).

```
   Transport  =  HOW the message travels    (stdio / HTTP / WebSocket)
   Primitive  =  WHAT the message carries    (Tools / Resources / Prompts / Sampling / Roots)

   Both use the SAME JSON-RPC 2.0 envelope.
```

### What we'll run

| Section | Demonstrates |
|---|---|
| **Setup** | Helpers + a mini in-process server → the JSON-RPC envelope |
| **Part A** | Three **transports**: stdio (real subprocess), HTTP, WebSocket |
| **Part B** | Five **primitives**: Tools, Resources, Prompts, Sampling, Roots |
| **Reference** | One-page cheat sheet — the whole picture on a card |

Every demo prints the **raw JSON-RPC messages** that travel on the wire, next to a plain-English
explanation — so beginners see exactly what crosses the boundary.

> **The theme is banking / financial services** throughout (`BankingMCPServer`), to keep the
> examples concrete for our audience.

> **Setup:** only the **Sampling** demo (Part B4) needs an API key. Create a `.env` with
> `ANTHROPIC_API_KEY=sk-ant-...` if you want real Claude there — every other cell runs without one.

## Setup — imports and (optional) API key

The whole notebook runs on the **Python standard library** plus two optional packages:

```
pip install python-dotenv anthropic   # both optional; only Sampling uses anthropic
```

Unlike a notebook that needs Claude for every step, here we **warn** instead of exiting when no
key is present — only the Sampling cell degrades to a mock.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import textwrap
from datetime import datetime
from typing import Any

# Optional: load a .env file if python-dotenv is installed.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

print("Core imports ready (standard library).")

In [ ]:
# Only the SAMPLING demo (Part B4) needs a real key. Everything else runs without one,
# so we WARN rather than exit.
API_KEY = os.getenv("ANTHROPIC_API_KEY") or os.getenv("CLAUDE_API_KEY", "")

# Valid IDs include: claude-haiku-4-5, claude-sonnet-4-6, claude-opus-4-8
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-4-6")

try:
    from anthropic import Anthropic
    anthropic_client = Anthropic(api_key=API_KEY) if API_KEY else None
except Exception:
    anthropic_client = None

if API_KEY and anthropic_client:
    print(f"API key found — the Sampling demo will call real Claude ({MODEL}).")
else:
    print("No API key found — the Sampling demo will use a mock response.")
    print("Every other cell in this notebook runs without a key.")

## The JSON-RPC 2.0 envelope

Everything in MCP is **JSON-RPC 2.0**. There are only two shapes to learn.

A **request** (client → server):
```json
{ "jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {} }
```

A **response** (server → client):
```json
{ "jsonrpc": "2.0", "id": 1, "result": { ... } }                                  // success
{ "jsonrpc": "2.0", "id": 1, "error": { "code": -32601, "message": "..." } }      // failure
```

The `id` ties a response back to its request. `method` names the operation; `params` carries its
arguments. **That's the entire contract** — the same envelope rides over every transport below.

These helpers build those shapes, and `show_step()` prints any message the way it looks on the wire.

In [ ]:
def _pretty(obj: Any) -> str:
    """Pretty-print any JSON-serialisable object (what you'd see on the wire)."""
    return json.dumps(obj, indent=2, default=str)


def jsonrpc_request(method: str, params: dict, req_id: int = 1) -> dict:
    """Build a well-formed JSON-RPC 2.0 REQUEST."""
    return {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params}


def jsonrpc_result(req_id: int, result: Any) -> dict:
    """Build a well-formed JSON-RPC 2.0 SUCCESS response."""
    return {"jsonrpc": "2.0", "id": req_id, "result": result}


def jsonrpc_error(req_id: int, code: int, message: str) -> dict:
    """Build a well-formed JSON-RPC 2.0 ERROR response."""
    return {"jsonrpc": "2.0", "id": req_id, "error": {"code": code, "message": message}}


def show_step(label: str, payload: dict) -> None:
    """Print one message exactly as it travels on the wire — the teaching centrepiece."""
    print(f"── {label} ──")
    print(_pretty(payload))
    print()


# A quick look at the two shapes:
show_step("Example REQUEST  (client -> server)", jsonrpc_request("tools/list", {}, 1))
show_step("Example RESPONSE (server -> client)", jsonrpc_result(1, {"tools": ["..."]}))

## A mini in-process MCP server

A real MCP server runs as its own program. To keep the **protocol visible** (and the notebook
dependency-free), we use a tiny `MiniMCPServer` that lives in this process and answers the real
MCP methods: `initialize`, `tools/list`, `tools/call`, `resources/list`, `resources/read`,
`prompts/list`, `prompts/get`, and `roots/list`.

One method — `handle(request)` — is the single JSON-RPC gateway a client talks to. Read it once
and the whole protocol is demystified: it's a `method` switch that returns a `result` or an `error`.

In [ ]:
class MiniMCPServer:
    """
    A tiny, self-contained MCP server that runs inside this notebook's process.

    It deliberately mimics the real JSON-RPC message shapes so you can see exactly what travels
    on the wire — without a subprocess or a network in the way. The single entry point a client
    talks to is handle().
    """

    def __init__(self, name: str = "MiniServer"):
        self.name = name
        self._tools: dict[str, dict] = {}
        self._resources: dict[str, Any] = {}
        self._prompts: dict[str, dict] = {}
        self._roots: list[dict] = []          # filled by the client (Roots primitive)

    # ── Registration helpers ─────────────────────────────────────────────────
    def register_tool(self, name, description, schema, fn):
        self._tools[name] = {"description": description, "inputSchema": schema, "fn": fn}

    def register_resource(self, uri, description, value):
        self._resources[uri] = {"description": description, "value": value}

    def register_prompt(self, name, description, template, args):
        self._prompts[name] = {"description": description, "template": template, "arguments": args}

    # ── The single JSON-RPC gateway ──────────────────────────────────────────
    def handle(self, request: dict) -> dict:
        method = request.get("method", "")
        rid = request.get("id", 0)
        params = request.get("params") or {}

        if method == "initialize":
            return jsonrpc_result(rid, {
                "protocolVersion": "2024-11-05",
                "serverInfo": {"name": self.name, "version": "1.0.0"},
                "capabilities": {"tools": {}, "resources": {}, "prompts": {}},
            })

        elif method == "tools/list":
            tools = [
                {"name": n, "description": d["description"], "inputSchema": d["inputSchema"]}
                for n, d in self._tools.items()
            ]
            return jsonrpc_result(rid, {"tools": tools})

        elif method == "tools/call":
            name = params.get("name", "")
            args = params.get("arguments", {})
            if name not in self._tools:
                return jsonrpc_error(rid, -32601, f"Tool '{name}' not found")
            try:
                result = self._tools[name]["fn"](**args)
                return jsonrpc_result(rid, {"content": [{"type": "text", "text": str(result)}], "isError": False})
            except Exception as exc:
                return jsonrpc_result(rid, {"content": [{"type": "text", "text": str(exc)}], "isError": True})

        elif method == "resources/list":
            resources = [
                {"uri": uri, "description": d["description"]}
                for uri, d in self._resources.items()
            ]
            return jsonrpc_result(rid, {"resources": resources})

        elif method == "resources/read":
            uri = params.get("uri", "")
            if uri not in self._resources:
                return jsonrpc_error(rid, -32601, f"Resource '{uri}' not found")
            value = self._resources[uri]["value"]
            text = _pretty(value) if not isinstance(value, str) else value
            return jsonrpc_result(rid, {"contents": [{"uri": uri, "mimeType": "text/plain", "text": text}]})

        elif method == "prompts/list":
            prompts = [
                {"name": n, "description": d["description"],
                 "arguments": [{"name": a, "required": True} for a in d["arguments"]]}
                for n, d in self._prompts.items()
            ]
            return jsonrpc_result(rid, {"prompts": prompts})

        elif method == "prompts/get":
            name = params.get("name", "")
            args = params.get("arguments", {})
            if name not in self._prompts:
                return jsonrpc_error(rid, -32601, f"Prompt '{name}' not found")
            filled = self._prompts[name]["template"]
            for k, v in args.items():
                filled = filled.replace(f"{{{k}}}", str(v))
            return jsonrpc_result(rid, {
                "description": self._prompts[name]["description"],
                "messages": [{"role": "user", "content": {"type": "text", "text": filled}}],
            })

        elif method == "roots/list":
            # Roots: the SERVER calls this to ask the CLIENT what it may access.
            return jsonrpc_result(rid, {"roots": self._roots})

        elif method == "sampling/createMessage":
            # Sampling: server requests the client's LLM — handled in the demo cell.
            return jsonrpc_result(rid, {"__sampling_request__": True, "params": params})

        else:
            return jsonrpc_error(rid, -32601, f"Unknown method: {method}")


print("MiniMCPServer defined.")

## Register the banking primitives

Now we populate one `BankingMCPServer` with a realistic financial-services surface:

- **Tools** — `get_account_balance`, `calculate_emi`
- **Resources** — `bank://policies/kyc-rules`, `bank://customers/C001/profile`
- **Prompts** — `summarize_account_activity`, `draft_loan_decision_letter`

The **same server instance** powers every demo from here on, so learners watch one server answer
many different kinds of request.

In [ ]:
server = MiniMCPServer("BankingMCPServer")

# ── Tools (actions the AI can trigger) ───────────────────────────────────────
server.register_tool(
    "get_account_balance",
    "Return the mock balance for a given account number.",
    {"type": "object",
     "properties": {"account_id": {"type": "string", "description": "Account number"}},
     "required": ["account_id"]},
    lambda account_id: f"Account {account_id}: Balance = INR {(hash(account_id) % 90000) + 10000:,}.00",
)
server.register_tool(
    "calculate_emi",
    "Calculate monthly EMI for a loan.",
    {"type": "object",
     "properties": {"principal": {"type": "number"},
                    "annual_rate": {"type": "number", "description": "Annual interest rate (%)"},
                    "months": {"type": "integer"}},
     "required": ["principal", "annual_rate", "months"]},
    lambda principal, annual_rate, months: (
        f"EMI = INR {principal * (annual_rate/1200) * (1 + annual_rate/1200)**months / ((1 + annual_rate/1200)**months - 1):,.2f} / month"
    ),
)

# ── Resources (read-only data the server exposes) ────────────────────────────
server.register_resource(
    "bank://policies/kyc-rules",
    "KYC compliance rules document",
    "KYC Rules v3.2: 1) Govt-issued photo ID required. 2) Address proof mandatory. "
    "3) PAN card for transactions > INR 50,000.",
)
server.register_resource(
    "bank://customers/C001/profile",
    "Customer profile for C001",
    {"id": "C001", "name": "Arjun Sharma", "tier": "Gold", "since": "2019-03-15", "branch": "Chennai-OMR"},
)

# ── Prompts (reusable templates the user picks) ──────────────────────────────
server.register_prompt(
    "summarize_account_activity",
    "Generate a summary prompt for a customer's recent account activity.",
    "Summarise the last {days} days of activity for account {account_id}. "
    "Highlight any unusual transactions, flag potential fraud, and recommend actions.",
    ["account_id", "days"],
)
server.register_prompt(
    "draft_loan_decision_letter",
    "Draft a formal loan decision letter.",
    "Draft a formal letter to customer {customer_name} regarding their loan application "
    "for INR {loan_amount}. Decision: {decision}. Be professional and concise.",
    ["customer_name", "loan_amount", "decision"],
)

print("BankingMCPServer ready.")
print("  Tools    :", list(server._tools))
print("  Resources:", list(server._resources))
print("  Prompts  :", list(server._prompts))

### Talk to the server by hand — the handshake

Before any transport or client wrapper, send the server **one raw request** yourself. The
`initialize` handshake begins every MCP session: the client says hello, the server announces its
name, version, protocol, and capabilities.

In [ ]:
init_req = jsonrpc_request("initialize",
                           {"protocolVersion": "2024-11-05", "clientInfo": {"name": "explorer"}}, 1)
print("REQUEST :", json.dumps(init_req))
print("RESPONSE:")
print(_pretty(server.handle(init_req)))

---
# Part A — The three **transports** (how the message travels)

A transport is just the **pipe**. The JSON-RPC message inside is identical every time — only the
delivery mechanism changes.

| Transport | Where's the server? | Direction | Best for |
|---|---|---|---|
| **stdio** | Same machine (subprocess) | Request → Reply | Local tools, Claude Desktop |
| **HTTP** | Remote URL | Request → Reply (can stream) | Hosted / shared services |
| **WebSocket** | Remote URL | Both sides, anytime | Real-time push, live alerts |

> **The golden rule:** pick the transport based on *where the server lives* and *how live the
> session needs to be*. The primitive you send is unaffected.

## A1 · stdio transport — passing notes in the same room

**The pipe is two text streams:** the client writes JSON to the server's **stdin**; the server
writes JSON back on its **stdout**. No network, no port, no auth. Fastest and simplest.

**Analogy:** two people in the same room passing written notes.

**Use it when** the server is local — file readers, local databases, code runners. *Claude Desktop
uses stdio for all its built-in servers.*
**Avoid it when** many clients must share one server across a network (→ HTTP), or you need the
server to push (→ WebSocket).

The next cell writes a **standalone** server script to disk and shows its full source — this is a
complete MCP server in ~30 lines.

In [ ]:
STDIO_SERVER_CODE = r"""
# stdio_datetime_server.py
# A STANDALONE MCP server: reads JSON-RPC from stdin, writes JSON-RPC to stdout.
# The client (this notebook) launches it as a subprocess -- they talk over pipes.

import sys, json
from datetime import datetime

TOOLS = [{
    "name": "get_current_datetime",
    "description": "Returns current date/time in the given strftime format.",
    "inputSchema": {
        "type": "object",
        "properties": {"date_format": {"type": "string"}},
        "required": ["date_format"],
    },
}]

def handle(req):
    m, rid, params = req.get("method"), req.get("id"), req.get("params") or {}
    if m == "initialize":
        res = {"protocolVersion": "2024-11-05",
               "serverInfo": {"name": "stdio-datetime", "version": "1.0.0"},
               "capabilities": {"tools": {}}}
    elif m == "tools/list":
        res = {"tools": TOOLS}
    elif m == "tools/call":
        fmt = (params.get("arguments") or {}).get("date_format", "%Y-%m-%d %H:%M:%S")
        res = {"content": [{"type": "text", "text": datetime.now().strftime(fmt)}], "isError": False}
    else:
        return {"jsonrpc": "2.0", "id": rid,
                "error": {"code": -32601, "message": "Unknown method"}}
    return {"jsonrpc": "2.0", "id": rid, "result": res}

for line in sys.stdin:               # blocks, waiting for each request line
    line = line.strip()
    if not line:
        continue
    sys.stdout.write(json.dumps(handle(json.loads(line))) + "\n")
    sys.stdout.flush()               # must flush or the pipe blocks
"""

with open("stdio_datetime_server.py", "w") as f:
    f.write(STDIO_SERVER_CODE)

print("Wrote stdio_datetime_server.py -- a real, standalone MCP server. Source:")
print(STDIO_SERVER_CODE)

Now the client side. The cell below **launches that script as a real subprocess** and runs the
full `initialize → tools/list → tools/call` handshake across the process boundary. Watch the wire
log — those bytes genuinely cross from one process to another over stdin/stdout.

> 🔧 **Interactive:** change `date_format` (any Python `strftime` string) and re-run.

In [ ]:
# ── Change me and re-run ──
date_format = "%d %B %Y - %I:%M %p"

proc = subprocess.Popen(
    [sys.executable, "stdio_datetime_server.py"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

def stdio_send(req):
    """Write one JSON-RPC line to the child's stdin, read one line back from its stdout."""
    show_step("-> CLIENT sends (stdin)", req)
    proc.stdin.write(json.dumps(req) + "\n")
    proc.stdin.flush()
    resp = json.loads(proc.stdout.readline())
    show_step("<- SERVER responds (stdout)", resp)
    return resp

# The full handshake, across a real process boundary:
stdio_send(jsonrpc_request("initialize",
                           {"protocolVersion": "2024-11-05", "clientInfo": {"name": "explorer"}}, 1))
stdio_send(jsonrpc_request("tools/list", {}, 2))
final = stdio_send(jsonrpc_request("tools/call",
                                   {"name": "get_current_datetime",
                                    "arguments": {"date_format": date_format}}, 3))

proc.stdin.close()
proc.wait(timeout=5)

print("RESULT:", final["result"]["content"][0]["text"])
print("\nSame JSON-RPC you saw in-process -- now across a real subprocess pipe.")

## A2 · HTTP transport — posting a letter to an address

The server lives at a **URL**. The client sends an HTTP **POST**; the server replies with a JSON
body. For long operations it can **stream** partial results with Server-Sent Events (SSE).

**Analogy:** posting a letter — you write the address (URL), seal the letter (the JSON-RPC body),
post it, and wait for the reply.

**Use it when** the server is hosted and shared — cloud APIs, SaaS integrations — behind standard
web infrastructure (load balancers, auth tokens, TLS). Many clients can share one server.

**Key point:** the JSON-RPC body is **identical** to what stdio sent — only the envelope around it
changes. The cell prints the full HTTP request and response so you can see that envelope wrapping
the same message.

> *We show the real HTTP envelope but process the body with our in-process server, so the focus
> stays on the message shape, not network plumbing.*
>
> 🔧 **Interactive:** change `account_id` and re-run.

In [ ]:
# ── Change me and re-run ──
account_id = "ACC-BANK-001"

# The client builds a normal JSON-RPC request...
req = jsonrpc_request("tools/call",
                      {"name": "get_account_balance", "arguments": {"account_id": account_id}}, 1)

# ...and wraps it in an HTTP POST envelope (the ONLY thing that differs from stdio).
http_request = textwrap.dedent(f"""\
    POST /mcp HTTP/1.1
    Host: api.mybank.com
    Content-Type: application/json
    Authorization: Bearer <token>

    {_pretty(req)}""")

# The same in-process server processes the body (same JSON-RPC, different pipe).
server.handle(jsonrpc_request("initialize", {}, 0))
resp = server.handle(req)

http_response = textwrap.dedent(f"""\
    HTTP/1.1 200 OK
    Content-Type: application/json

    {_pretty(resp)}""")

print("== HTTP REQUEST  (client -> server) ==")
print(http_request)
print("\n== HTTP RESPONSE (server -> client) ==")
print(http_response)
print("\nRESULT:", resp["result"]["content"][0]["text"])

## A3 · WebSocket transport — a phone call left off the hook

**One persistent connection** stays open for the whole session. Both sides can send **at any
time** — the server can **push** without being asked.

**Analogy:** a phone call left off the hook — either person speaks whenever they want, no
redialling.

**Use it when** you need bidirectional, real-time messaging: live market data, streaming tool
output, real-time fraud alerts, event-driven agents.

Watch for the **PUSH** message in the log below — the server sends a `notifications/...` message
with **no request** from the client. stdio and HTTP can't do that; the client always has to ask
first. That push is the one feature unique to WebSockets — everything else is still the same
JSON-RPC envelope.

> 🔧 **Interactive:** change `topic` (used in the server's push notification) and re-run.

In [ ]:
# ── Change me and re-run ──
topic = "FX rates"

conversation = []
def ws_log(direction, payload):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    conversation.append((ts, direction, payload))

# Client connects and initialises over ONE persistent connection.
init_req = jsonrpc_request("initialize",
                           {"protocolVersion": "2024-11-05", "clientInfo": {"name": "ws-explorer"}}, 1)
ws_log("CLIENT -> SERVER", init_req)
ws_log("SERVER -> CLIENT", server.handle(init_req))

# Client discovers tools.
list_req = jsonrpc_request("tools/list", {}, 2)
ws_log("CLIENT -> SERVER", list_req)
ws_log("SERVER -> CLIENT", server.handle(list_req))

# * Server PUSHES a notification with NO request -- only WebSockets can do this.
ws_log("SERVER -> CLIENT  (PUSH - no request!)", {
    "jsonrpc": "2.0",
    "method": "notifications/tools/list_changed",
    "params": {"message": f"New tool added related to '{topic}' - please re-list!"},
})

# Client calls a tool.
call_req = jsonrpc_request("tools/call",
                           {"name": "get_account_balance", "arguments": {"account_id": "ACC-WS-001"}}, 3)
ws_log("CLIENT -> SERVER", call_req)
ws_log("SERVER -> CLIENT", server.handle(call_req))

for ts, direction, payload in conversation:
    print(f"[{ts}]  {direction}")
    print(_pretty(payload))
    print()

---
# Part B — The five **primitives** (what the message carries)

If the transport is the pipe, the primitives are what flows through it. Each has a **direction**
and a **controller** — *who decides* it happens — and that's the thing beginners most need to keep
straight.

| # | Primitive | Direction | Controlled by | One-liner | Analogy |
|---|---|---|---|---|---|
| 1 | **Tools** | client → server | 🤖 Model | "Do this action" | Buttons the AI presses |
| 2 | **Resources** | client → server | 🖥️ App | "Read this data" | Files in a shared folder |
| 3 | **Prompts** | client → server | 👤 User | "Run this template" | Saved email templates |
| 4 | **Sampling** | server → client | 🔄 Server asks | "Borrow your AI" | Contractor uses your phone |
| 5 | **Roots** | server → client | 🔒 Client declares | "Here's your fence" | Keys to specific rooms |

All five ride the same JSON-RPC envelope you already know — we just call different `method`s.

## B1 · Tools — the VERBS of MCP *(model-controlled)*

Tools are **actions** the AI can trigger.

**How it works:** the client calls `tools/list` → the server returns every tool's name + JSON
Schema. The **model** (Claude) reads those schemas and decides *which* tool to call and *what*
arguments to pass. The client then calls `tools/call` → the server runs the function and returns
the result.

**Why "model-controlled"?** The model decides when to call a tool and with which arguments — the
human doesn't pick.

**Banking examples:** `get_account_balance(account_id)`, `transfer_funds(...)`,
`flag_transaction_as_fraud(txn_id)`, `calculate_emi(principal, rate, months)`.

**Key rule:** tools *do* things (often with side effects). If you only want to *read* data, that's
a Resource (next).

> 🔧 **Interactive:** change `account_id`, `principal`, `annual_rate`, `months` and re-run.

In [ ]:
# ── Change me and re-run ──
account_id = "ACC-JPM-2025"
principal, annual_rate, months = 500000, 8.5, 60

# Step 1 - discover what tools exist.
list_req = jsonrpc_request("tools/list", {}, 1)
show_step("tools/list  REQUEST", list_req)
show_step("tools/list  RESPONSE", server.handle(list_req))

# Step 2 - call the balance tool.
bal_req = jsonrpc_request("tools/call",
                          {"name": "get_account_balance", "arguments": {"account_id": account_id}}, 2)
bal_resp = server.handle(bal_req)
show_step("tools/call -> get_account_balance", bal_req)
show_step("tools/call <- result", bal_resp)

# Step 3 - call the EMI tool.
emi_req = jsonrpc_request("tools/call",
                          {"name": "calculate_emi",
                           "arguments": {"principal": principal, "annual_rate": annual_rate, "months": months}}, 3)
emi_resp = server.handle(emi_req)
show_step("tools/call -> calculate_emi", emi_req)
show_step("tools/call <- result", emi_resp)

print("RESULT:")
print(" ", bal_resp["result"]["content"][0]["text"])
print(" ", emi_resp["result"]["content"][0]["text"])

## B2 · Resources — the NOUNS of MCP *(app-controlled)*

Resources are **read-only data** the server exposes via URIs.

**How it works:** the client calls `resources/list` → server returns URIs + descriptions. The
client calls `resources/read` with a URI → server returns the content, which is injected into the
AI's context (not "called" like a tool).

**Why "app-controlled"?** The **application** decides which resources to fetch and when — the AI
reads passively and can't trigger a resource load itself.

**URIs look like paths or web addresses:** `bank://policies/kyc-rules`,
`bank://customers/C001/profile`, `db://transactions/last-30-days`.

**Tools vs Resources:** tools = *do* something (verb, side effects possible); resources = *read*
something (noun, always read-only).

> 🔧 **Interactive:** switch `resource_uri` between the two registered URIs and re-run.

In [ ]:
# ── Change me and re-run ──
resource_uri = "bank://policies/kyc-rules"   # or "bank://customers/C001/profile"

list_req = jsonrpc_request("resources/list", {}, 1)
show_step("resources/list  REQUEST", list_req)
show_step("resources/list  RESPONSE", server.handle(list_req))

read_req = jsonrpc_request("resources/read", {"uri": resource_uri}, 2)
read_resp = server.handle(read_req)
show_step("resources/read  REQUEST", read_req)
show_step("resources/read  RESPONSE", read_resp)

content = read_resp["result"]["contents"][0]["text"]
print("RESOURCE CONTENT (this is injected into the AI's context):\n")
print(content)

## B3 · Prompts — reusable TEMPLATES *(user-controlled)*

Prompts are saved, fill-in-the-blank templates — think slash-commands.

**How it works:** the client calls `prompts/list` → server returns the templates. The **user**
picks one (e.g. `/summarize-account`). The client calls `prompts/get` with the chosen name +
arguments → the server fills the `{placeholders}` and returns ready-to-send messages.

**Why "user-controlled"?** The **user** chooses which prompt to run — unlike Tools (model picks)
or Resources (app picks). In Claude Desktop these appear as slash-commands.

**Analogy:** saved email templates — pick the template, fill in `{name}` and `{amount}`, done. No
re-typing the structure each time.

**Key benefit:** standardises *how* tasks are asked across a team, so everyone follows the same
best-practice prompt.

> 🔧 **Interactive:** try `prompt_name = "draft_loan_decision_letter"` with
> `arguments = {"customer_name": "Arjun Sharma", "loan_amount": "2500000", "decision": "Approved"}`.

In [ ]:
# ── Change me and re-run ──
prompt_name = "summarize_account_activity"
arguments = {"account_id": "ACC-JPM-2025", "days": "30"}

list_req = jsonrpc_request("prompts/list", {}, 1)
show_step("prompts/list  REQUEST", list_req)
show_step("prompts/list  RESPONSE", server.handle(list_req))

get_req = jsonrpc_request("prompts/get", {"name": prompt_name, "arguments": arguments}, 2)
get_resp = server.handle(get_req)
show_step("prompts/get  REQUEST", get_req)
show_step("prompts/get  RESPONSE", get_resp)

filled = get_resp["result"]["messages"][0]["content"]["text"]
print("FILLED PROMPT (ready to send to Claude):\n")
print(filled)

## B4 · Sampling — the server BORROWS the client's AI *(reverse direction!)*

Sampling lets the **server** ask the **client's** LLM to do some reasoning, instead of shipping
its own model.

**The flow (a round-trip inside a round-trip):**
1. Client calls a tool on the server *(normal, left-to-right)*.
2. Server needs AI reasoning → sends `sampling/createMessage` **back to the client**.
3. Client receives it and calls **its own** Claude with those messages.
4. Client returns Claude's answer to the server.
5. Server uses that answer to finish the original tool call.

**Analogy:** you hire a contractor (server) to write a report. Mid-way they say *"I need a lawyer
to check one clause — can you call yours?"* You make the call and relay the answer; they finish
the report.

**Why it matters:** the server ships **zero** model config, **zero** API key, **zero** billing.
The same server runs in Claude Desktop, an IDE, or your app — each host supplies its own model.
The server describes the *task*; the client owns the *model*.

**Security:** the client always controls whether to fulfil a sampling request — it can refuse,
rate-limit, or substitute a different model.

> *With `ANTHROPIC_API_KEY` set, this cell calls real Claude. Otherwise it prints a mock so the
> flow still runs.*
>
> 🔧 **Interactive:** change `text_to_summarize` and re-run.

In [ ]:
# ── Change me and re-run ──
text_to_summarize = (
    "The Model Context Protocol (MCP) standardises how AI applications connect to external tools "
    "and data sources. It uses JSON-RPC 2.0 and defines five primitives: Tools, Resources, Prompts, "
    "Sampling, and Roots. Any MCP-compatible client can work with any MCP-compatible server without "
    "custom integration code."
)

# Step 1 - client calls a tool on the server (normal, left-to-right).
call_req = jsonrpc_request("tools/call",
                           {"name": "get_account_balance", "arguments": {"account_id": "ACC-SAMPLE"}}, 1)
show_step("Step 1 - Client calls a tool", call_req)

# Step 2 - server needs AI reasoning, so it sends a sampling request BACK to the client.
sampling_request = {
    "jsonrpc": "2.0", "id": 99, "method": "sampling/createMessage",
    "params": {
        "messages": [{"role": "user", "content": {"type": "text",
                       "text": f"Please summarise the following in one sentence:\n\n{text_to_summarize}"}}],
        "maxTokens": 200,
        "systemPrompt": "You are a concise financial analyst. Summarise clearly.",
    },
}
show_step("Step 2 - Server asks the CLIENT's LLM (reverse direction!)", sampling_request)

# Step 3 - client fulfils it with ITS OWN Claude (or a mock if no key).
if anthropic_client and API_KEY:
    try:
        resp = anthropic_client.messages.create(
            model=MODEL, max_tokens=200,
            messages=[{"role": "user",
                       "content": f"Please summarise the following in one sentence:\n\n{text_to_summarize}"}],
        )
        ai_answer = resp.content[0].text
    except Exception as exc:
        ai_answer = f"[API call failed: {exc} - using mock] Summary of: '{text_to_summarize[:60]}...'"
else:
    ai_answer = f"[No API key - mock] The text explains MCP: '{text_to_summarize[:50]}...'"

sampling_response = jsonrpc_result(99, {
    "role": "assistant",
    "content": {"type": "text", "text": ai_answer},
    "model": MODEL, "stopReason": "end_turn",
})
show_step("Step 3 - Client returns Claude's answer to the server", sampling_response)

# Step 4 - server completes the original tool call using the AI output.
final_result = jsonrpc_result(1, {
    "content": [{"type": "text", "text": f"Summary (produced by client's Claude): {ai_answer}"}],
    "isError": False,
})
show_step("Step 4 - Server returns the final tool result", final_result)

print("AI SUMMARY:\n", ai_answer)

## B5 · Roots — the client sets the SCOPE FENCE *(client-controlled)*

Roots are the boundaries the **client** declares for the server — which folders or URLs it may
operate within.

**How it works:** before connecting, the client sets a list of allowed URIs. The server can call
`roots/list` to ask *"what am I allowed to access?"*. A well-behaved server stays inside those
roots. The client is the **sole authority**.

**Analogy:** you hire a cleaner (server) and give them keys to the living room and kitchen — but
not the bedroom or office. They can only work where the keys fit; you decide which keys they get.

**Why it matters — security:** least privilege by design. A filesystem server can't read
`/etc/passwords` if it's not in the roots; a GitHub server can't touch repos outside the declared
list. Security lives on the **client** side, where the human or company policy controls it.

**The pattern:** *capability lives on the server; authority lives on the client.*

> 🔧 **Interactive:** change `root1` / `root2` and re-run to see the granted fence change.

In [ ]:
# ── Change me and re-run ──
root1 = "file:///home/user/project"
root2 = "file:///home/user/shared"

# The CLIENT declares the allowed roots before the server does anything.
declared_roots = [r.strip() for r in [root1, root2] if r.strip()]
server._roots = [{"uri": r, "name": r.split("/")[-1] or r} for r in declared_roots]

# The server asks: "what am I allowed to touch?"
roots_req = jsonrpc_request("roots/list", {}, 1)
show_step("Server asks the client: roots/list", roots_req)
show_step("Client answers with its declared roots", server.handle(roots_req))

# A path INSIDE a declared root -> allowed.
allowed = declared_roots[0] if declared_roots else "none"
show_step("Server checks a path INSIDE its roots", {
    "server_attempted_to_access": f"{allowed}/transactions/2025.csv",
    "is_within_roots": True,
    "action": "ALLOWED - path is inside a declared root",
})

# A path OUTSIDE every root -> denied.
show_step("Server checks a path OUTSIDE its roots", {
    "server_attempted_to_access": "/etc/passwords",
    "is_within_roots": False,
    "action": "DENIED - path is NOT in any declared root",
})

print("Roots granted to the server:")
for r in declared_roots:
    print("  [OK]", r)
print("\nCapability lives on the server. Authority lives on the client.")

---
## 📋 Quick reference — one-page cheat sheet

### 🔌 Three transports (the PIPE)

| Transport | Where's the server? | Direction | Best for |
|---|---|---|---|
| **stdio** | Same machine (subprocess) | Request → Reply | Local tools, Claude Desktop |
| **HTTP** | Remote URL | Request → Reply (can stream) | Hosted / shared services |
| **WebSocket** | Remote URL | Both sides, anytime | Real-time push, live alerts |

> The JSON-RPC 2.0 message is **identical** in all three — only how it travels changes.

### 🧩 Five primitives (what FLOWS through the pipe)

| # | Primitive | Direction | Controlled by | One-liner | Analogy |
|---|---|---|---|---|---|
| 1 | **Tools** | client → server | 🤖 Model | "Do this action" | Buttons the AI presses |
| 2 | **Resources** | client → server | 🖥️ App | "Read this data" | Files in a shared folder |
| 3 | **Prompts** | client → server | 👤 User | "Run this template" | Saved email templates |
| 4 | **Sampling** | server → client | 🔄 Server asks | "Borrow your AI" | Contractor uses your phone |
| 5 | **Roots** | server → client | 🔒 Client declares | "Here's your fence" | Keys to specific rooms |

### 🏦 Banking examples

```
Tools     ->  get_account_balance(account_id)
              flag_transaction_as_fraud(txn_id)
              calculate_emi(principal, rate, months)

Resources ->  bank://policies/kyc-rules
              bank://customers/C001/profile
              bank://rates/current

Prompts   ->  /summarize-account-activity {account_id} {days}
              /draft-loan-decision-letter {name} {amount} {decision}

Sampling  ->  Server asks: "Summarise this transaction narrative for me"
              Client answers with Claude - server has no API key of its own

Roots     ->  file:///home/analyst/approved-reports   [OK]  allowed
              /etc/passwords                           [X]   not in roots - denied
```

### 📐 The golden rule

```
Transport = HOW the message travels
Primitive = WHAT the message carries

Both use the same JSON-RPC 2.0 envelope.
Pick the transport based on where the server lives and how live the session must be.
```

---

**Where to go next:** the companion notebook `mcp_client_server_demo.ipynb` wires these same
primitives into Claude's tool-use loop and swaps our `MiniMCPServer` for the official **`mcp`
Python SDK** (`FastMCP` + `ClientSession`) running real servers over live subprocesses.

**Homework for the class:** register a third tool on `BankingMCPServer` —
`flag_transaction_as_fraud(transaction_id)` — then re-run the **B1 · Tools** cell. It appears in
`tools/list` with **zero** changes to any transport, because tools are discovered at runtime.

🎉 **You've now seen all three transports and all five primitives — the complete MCP surface —
running cell by cell.**